In [ ]:
rows = 5
columns = 8

startStates= [(5,c) for c in range (2,9)]

destination = (3,1)
dangerPosition = [(5,1), (1,1)]

reward = 1000
penalty = -1000
noise= 0.1
gamma = 0.99

In [ ]:
directions = {

           "N": (1,0),
           "NW": (1, -1),
           "NE": (1, 1),
           "S": (-1, 0),
           "SW": (-1, -1),
           "SE": (-1, 1)
}

In [ ]:
positions = [(row, col) for row in range(1, rows + 1)
                  for col in range(1, columns + 1)]

print("Total states:", len(positions))

Total states: 40


In [ ]:
def move(currentState, chosenDirection):
    row, col = currentState
    dr, dc = directions[chosenDirection]

    newRow = row + dr
    newCol = col + dc

    if newRow < 1 or newRow > rows:
        return currentState

    if newCol < 1 or newCol > columns:
        return currentState

    return (newRow, newCol)

In [ ]:
print(move((3, 3), "N"))
print(move((5, 3), "N"))
print(move((3, 1), "NW"))

(4, 3)
(5, 3)
(3, 1)


In [ ]:
mainProbability = 1 - noise
otherprobability = noise / 4

print("Intended:", mainProbability)
print("Unintended:", otherprobability)

Intended: 0.9
Unintended: 0.025


In [ ]:
otherDirections = {
    "N": ["NE", "SE", "SW", "NW"],
    "NE": ["SE", "S", "NW", "N"],
    "SE": ["S", "SW", "N", "NE"],
    "S": ["SW", "NW", "NE", "SE"],
    "SW": ["NW", "N", "S", "SE"],
    "NW": ["N", "NE", "S", "SW"]
}

In [ ]:
def transitions(currentState, chosenDirection):
    probabilities = {}

    possibleMoves = [chosenDirection] + otherDirections[chosenDirection]

    for index, direction in enumerate(possibleMoves):
        nextState = move(currentState, direction)

        if index == 0:
            probability = mainProbability
        else:
            probability = otherprobability

        probabilities[nextState] = probabilities.get(nextState, 0) + probability

    return probabilities

In [ ]:
print(transitions((3, 3), "N"))
print(sum(transitions((3, 3), "N").values()))

{(4, 3): 0.9, (4, 4): 0.025, (2, 4): 0.025, (2, 2): 0.025, (4, 2): 0.025}
1.0


In [ ]:
def getReward(currentState, moveReward):
    if currentState == destination:
        return reward

    if currentState in dangerPosition:
        return penalty

    return moveReward

In [ ]:
print(getReward((3, 1), -20))
print(getReward((5, 1), -20))
print(getReward((3, 3), -20))

1000
-1000
-20


In [ ]:
utilityValues = {position: 0 for position in positions}

maxIterations = 500

print("Total utilities:", len(utilityValues))

Total utilities: 40


In [ ]:
def calculateIteration(moveReward):
    values = {position: 0 for position in positions}

    for position in positions:
        if position == destination or position in dangerPosition:
            values[position] = getReward(position, moveReward)

    for i in range(maxIterations):
        newValues = values.copy()

        for position in positions:
            if position == destination or position in dangerPosition:
                continue

            actionValues = []

            for direction in directions:
                expectedValue = 0

                for nextState, probability in transitions(position, direction).items():
                    expectedValue += probability * values[nextState]

                actionValues.append(expectedValue)

            newValues[position] = getReward(position, moveReward) + gamma * max(actionValues)

        values = newValues

    return values

In [ ]:
finalvalues = calculateIteration(-20)

print("Parking destination:", finalvalues[destination])
print("Danger position:", finalvalues[(5, 1)])
print("Normal position:", finalvalues[(4, 1)])

Parking destination: 1000
Danger position: -1000
Normal position: 964.8335128882969


In [ ]:
for moveReward in range(-20, 1):
    finalvalues = calculateIteration(moveReward)
    print("Reward:", moveReward, "Utility:", round(finalvalues[(4, 1)], 2))

Reward: -20 Utility: 964.83
Reward: -19 Utility: 966.0
Reward: -18 Utility: 967.17
Reward: -17 Utility: 968.34
Reward: -16 Utility: 969.51
Reward: -15 Utility: 970.68
Reward: -14 Utility: 971.84
Reward: -13 Utility: 973.01
Reward: -12 Utility: 974.18
Reward: -11 Utility: 975.35
Reward: -10 Utility: 976.52
Reward: -9 Utility: 977.69
Reward: -8 Utility: 978.86
Reward: -7 Utility: 980.03
Reward: -6 Utility: 981.19
Reward: -5 Utility: 982.36
Reward: -4 Utility: 983.53
Reward: -3 Utility: 984.7
Reward: -2 Utility: 985.87
Reward: -1 Utility: 987.05
Reward: 0 Utility: 988.22


In [ ]:
def getPolicy(finalvalues):
    policy = {}

    for position in positions:
        if position == destination or position in dangerPosition:
            continue

        bestDirection = None
        bestValue = float("-inf")

        for direction in directions:
            expectedValue = sum(
                probability * finalvalues[nextState]
                for nextState, probability in transitions(position, direction).items()
            )

            if expectedValue > bestValue:
                bestValue = expectedValue
                bestDirection = direction

        policy[position] = bestDirection

    return policy

In [ ]:
finalvalues = calculateIteration(-20)
parkingPolicy = getPolicy(finalvalues)

for start in startStates:
    print(start, "→", parkingPolicy[start])

(5, 2) → SW
(5, 3) → SW
(5, 4) → SW
(5, 5) → SW
(5, 6) → SW
(5, 7) → SW
(5, 8) → SW


In [ ]:
for start in startStates:
    currentState = start
    path = [currentState]

    for i in range(40):
        if currentState == destination or currentState in dangerPosition:
            break

        chosenDirection = parkingPolicy[currentState]
        currentState = move(currentState, chosenDirection)
        path.append(currentState)

    print(start, "→", path)

(5, 2) → [(5, 2), (4, 1), (3, 1)]
(5, 3) → [(5, 3), (4, 2), (3, 1)]
(5, 4) → [(5, 4), (4, 3), (3, 2), (4, 1), (3, 1)]
(5, 5) → [(5, 5), (4, 4), (3, 3), (4, 2), (3, 1)]
(5, 6) → [(5, 6), (4, 5), (3, 4), (4, 3), (3, 2), (4, 1), (3, 1)]
(5, 7) → [(5, 7), (4, 6), (3, 5), (4, 4), (3, 3), (4, 2), (3, 1)]
(5, 8) → [(5, 8), (4, 7), (3, 6), (4, 5), (3, 4), (4, 3), (3, 2), (4, 1), (3, 1)]


In [ ]:
successfulCars = 0

for start in startStates:
    currentState = start

    for i in range(40):
        if currentState == destination or currentState in dangerPosition:
            break

        currentState = move(currentState, parkingPolicy[currentState])

    if currentState == destination:
        successfulCars += 1

print("Successfully parked cars:", successfulCars)

Successfully parked cars: 7


In [ ]:
for moveReward in range(-20, 1):
    finalvalues = calculateIteration(moveReward)
    parkingPolicy = getPolicy(finalvalues)
    successfulCars = 0

    for start in startStates:
        currentState = start

        for i in range(40):
            if currentState == destination or currentState in dangerPosition:
                break

            currentState = move(currentState, parkingPolicy[currentState])

        if currentState == destination:
            successfulCars += 1

    print("Reward:", moveReward, "Parked cars:", successfulCars)

Reward: -20 Parked cars: 7
Reward: -19 Parked cars: 7
Reward: -18 Parked cars: 7
Reward: -17 Parked cars: 7
Reward: -16 Parked cars: 7
Reward: -15 Parked cars: 7
Reward: -14 Parked cars: 7
Reward: -13 Parked cars: 7
Reward: -12 Parked cars: 7
Reward: -11 Parked cars: 7
Reward: -10 Parked cars: 7
Reward: -9 Parked cars: 7
Reward: -8 Parked cars: 7
Reward: -7 Parked cars: 7
Reward: -6 Parked cars: 7
Reward: -5 Parked cars: 7
Reward: -4 Parked cars: 7
Reward: -3 Parked cars: 7
Reward: -2 Parked cars: 7
Reward: -1 Parked cars: 7
Reward: 0 Parked cars: 7


In [ ]:
bestReward = -20
bestValues = calculateIteration(bestReward)
bestPolicy = getPolicy(bestValues)

print("Best reward:", bestReward)
print("Policy created:", len(bestPolicy), "cells")

Best reward: -20
Policy created: 37 cells


In [ ]:
arrows = {
    "N": "↑", "NW": "↖", "NE": "↗",
    "S": "↓", "SW": "↙", "SE": "↘"
}

for row in range(rows, 0, -1):
    for col in range(1, columns + 1):
        position = (row, col)

        if position == destination:
            print(" G ", end=" ")
        elif position in dangerPosition:
            print(" X ", end=" ")
        else:
            print(" " + arrows[bestPolicy[position]] + " ", end=" ")
    print()

 X   ↙   ↙   ↙   ↙   ↙   ↙   ↙  
 ↓   ↙   ↙   ↙   ↙   ↙   ↙   ↙  
 G   ↖   ↖   ↖   ↖   ↖   ↖   ↖  
 ↑   ↖   ↖   ↖   ↖   ↖   ↖   ↖  
 X   ↖   ↖   ↖   ↖   ↖   ↖   ↖  


In [ ]:
import random

def testParking(start, parkingPolicy):
    currentState = start

    for i in range(100):
        if currentState == destination:
            return True

        if currentState in dangerPosition:
            return False

        chosenDirection = parkingPolicy[currentState]
        probabilities = transitions(currentState, chosenDirection)

        currentState = random.choices(
            list(probabilities.keys()),
            weights=list(probabilities.values())
        )[0]

    return currentState == destination

In [ ]:
for start in startStates:
    successfulCars = 0

    for i in range(100):
        if testParking(start, bestPolicy):
            successfulCars += 1

    print(start, ":", successfulCars, "out of 100 successful")

(5, 2) : 100 out of 100 successful
(5, 3) : 93 out of 100 successful
(5, 4) : 99 out of 100 successful
(5, 5) : 97 out of 100 successful
(5, 6) : 99 out of 100 successful
(5, 7) : 96 out of 100 successful
(5, 8) : 99 out of 100 successful


In [ ]:
problem = (3, 4)

print("New obstacle position:", problem)

New obstacle position: (3, 4)


In [ ]:
def move(currentState, chosenDirection):
    row, col = currentState
    dr, dc = directions[chosenDirection]

    newRow = row + dr
    newCol = col + dc

    if newRow < 1 or newRow > rows:
        return currentState

    if newCol < 1 or newCol > columns:
        return currentState

    if (newRow, newCol) == problem:
        return currentState

    return (newRow, newCol)

In [ ]:
print(move((4, 3), "SE"))
print(move((4, 5), "SW"))
print(move((4, 3), "SW"))

(4, 3)
(4, 5)
(3, 2)


In [ ]:
finalvalues2 = calculateIteration(-20)
parkingPolicy2 = getPolicy(finalvalues2)

print("R2 policy created:", len(parkingPolicy2), "cells")

R2 policy created: 37 cells


In [ ]:
for row in range(rows, 0, -1):
    for col in range(1, columns + 1):
        position = (row, col)

        if position == destination:
            print(" G ", end=" ")
        elif position in dangerPosition:
            print(" X ", end=" ")
        elif position == problem:
            print(" ■ ", end=" ")
        else:
            print(" " + arrows[parkingPolicy2[position]] + " ", end=" ")
    print()

 X   ↙   ↙   ↙   ↙   ↙   ↙   ↙  
 ↓   ↙   ↙   ↙   ↖   ↙   ↙   ↙  
 G   ↖   ↖   ■   ↖   ↖   ↖   ↖  
 ↑   ↖   ↖   ↖   ↙   ↖   ↖   ↖  
 X   ↖   ↖   ↖   ↖   ↖   ↖   ↖  


In [ ]:
differentCells = []

for position in positions:
    if position == destination or position in dangerPosition or position == problem:
        continue

    if bestPolicy[position] != parkingPolicy2[position]:
        differentCells.append(position)

print("Number of changed directions:", len(differentCells))
print("Changed positions:", differentCells)

Number of changed directions: 2
Changed positions: [(2, 5), (4, 5)]


In [ ]:
for moveReward in range(-20, 1):
    finalvalues2 = calculateIteration(moveReward)
    parkingPolicy2 = getPolicy(finalvalues2)
    successfulCars = 0

    for start in startStates:
        currentState = start

        for i in range(40):
            if currentState == destination or currentState in dangerPosition:
                break

            currentState = move(currentState, parkingPolicy2[currentState])

        if currentState == destination:
            successfulCars += 1

    print("Reward:", moveReward, "Parked cars:", successfulCars)

Reward: -20 Parked cars: 7
Reward: -19 Parked cars: 7
Reward: -18 Parked cars: 7
Reward: -17 Parked cars: 7
Reward: -16 Parked cars: 7
Reward: -15 Parked cars: 7
Reward: -14 Parked cars: 7
Reward: -13 Parked cars: 7
Reward: -12 Parked cars: 7
Reward: -11 Parked cars: 7
Reward: -10 Parked cars: 7
Reward: -9 Parked cars: 7
Reward: -8 Parked cars: 7
Reward: -7 Parked cars: 7
Reward: -6 Parked cars: 7
Reward: -5 Parked cars: 7
Reward: -4 Parked cars: 7
Reward: -3 Parked cars: 7
Reward: -2 Parked cars: 7
Reward: -1 Parked cars: 7
Reward: 0 Parked cars: 7


In [ ]:
bestReward2 = -20
finalvalues2 = calculateIteration(bestReward2)
parkingPolicy2 = getPolicy(finalvalues2)

print("R2 best reward:", bestReward2)
print("R2 policy created:", len(parkingPolicy2), "cells")

R2 best reward: -20
R2 policy created: 37 cells


In [ ]:
changedPositions = []

for position in positions:
    if position == destination or position in dangerPosition or position == problem:
        continue

    if bestPolicy[position] != parkingPolicy2[position]:
        changedPositions.append(position)

print("Changed positions:", changedPositions)

for position in changedPositions:
    print(position, "R1:", bestPolicy[position], "R2:", parkingPolicy2[position])

Changed positions: [(2, 5), (4, 5)]
(2, 5) R1: NW R2: SW
(4, 5) R1: SW R2: NW


In [ ]:
for start in startStates:
    successfulCars = 0

    for i in range(100):
        if testParking(start, parkingPolicy2):
            successfulCars += 1

    print(start, ":", successfulCars, "out of 100 successful")

(5, 2) : 100 out of 100 successful
(5, 3) : 100 out of 100 successful
(5, 4) : 99 out of 100 successful
(5, 5) : 98 out of 100 successful
(5, 6) : 100 out of 100 successful
(5, 7) : 98 out of 100 successful
(5, 8) : 99 out of 100 successful
